# 04 · Variation over time, and pH

Puts every processed take on one **lab-clock** axis (hours, days), loads a pH time series, aligns the
two and compares them.

Two clocks are involved (recorder, pH logger). Put their offsets in the config
(`recorder_clock_offset_s`, `ph: clock_offset_s`). A wrong offset shifts one curve against the other and
changes every correlation below. Check it with a known event (the acid addition is usually the easiest one).

In [ ]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
CHANNELS = None             # None = every channel any take has; or e.g. [1, 2]
BAND = "audio"
DT_S = 10.0                 # time bin for the timeline [s]
BASELINE_TAKE = "DEMO_002"  # take + window used as the quiet reference for EXCESS power
BASELINE = (5.0, 34.0)
PH_WINDOW_S = 10.0          # acoustic average around each pH sample [s]
EVENT_TIMES = {"acid": "2026-10-01 10:02:40"}   # lab-clock times to mark (from the lab book)

In [ ]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

In [ ]:
feats = za.load_all_features(cfg)
za.timeline.takes_overview(feats)

In [ ]:
chans = CHANNELS or sorted({c for f in feats for c in f.channels})
series = {}
for c in chans:
    s = za.timeline.level_timeline(feats, c, BAND, dt=DT_S)
    if s.notna().any():
        series[c] = s
lab = {c: next(f.label(c) for f in feats if c in f.channels) for c in series}
print({c: lab[c] for c in series})

## Load pH

The file is described in the `ph:` section of the config. Auto-detection handles a timestamp column,
separate `Date` + `Time` columns, or elapsed time plus a start. If it guesses wrong, name the columns
explicitly there.

In [ ]:
ph = za.ph.load_ph_from_config(cfg)
print(len(ph), "pH samples from", ph.index[0], "to", ph.index[-1])
ph.head()

## Timeline: acoustic level per channel, trigger rate, pH (stacked, shared time axis)

Each quantity gets its own panel. There is no twin y-axis on purpose: two scales on one plot invite
reading a relation into the choice of axis limits.

In [ ]:
marks = {k: pd.Timestamp(v) for k, v in EVENT_TIMES.items()}
panels = {f"{lab[c]}: {BAND} level [dB re FS²]": [(lab[c], dsp.db(s), zp.ch_color(c))] for c, s in series.items()}
c0 = chans[0]
rate = za.timeline.rate_timeline(feats, c0, BAND, bin_s=DT_S)
if rate.notna().any():
    panels[f"{lab[c0]}: trigger rate [1/s]"] = [("rate", rate, zp.ch_color(c0))]
extra = [c for c in ph.columns if c != "pH"][:1]
fig = zp.plot_timeline(panels, ph=ph, markers=marks, extra_ph_cols=extra,
                       title=f"{cfg.project}: {BAND} band, {DT_S:g} s bins, all takes")
zp.save(fig, cfg, "timeline_all")

## Excess power

Subtract the quiet baseline (in **linear** power) before comparing with pH. The raw level contains the
noise floor and any apparatus tone. Those add a constant that flattens every trend near the floor (in the
demo it moves the slope from −10.5 to −8.9 dB per pH unit).

In [ ]:
fb = za.load_features(cfg, BASELINE_TAKE)
excess = {}
for c, s in series.items():
    if c in fb.channels:
        base = fb.band_level_in_window(c, BAND, *BASELINE)
        excess[c] = (s - base).clip(lower=1e-30)
        print(f"{lab[c]}: baseline {dsp.db(base):.1f} dB re FS²")

## Align with pH and compare

In [ ]:
t_first = min(marks.values()) if marks else ph.index[0]
ph_use = ph[ph.index > t_first + pd.Timedelta(seconds=20)]   # after the reaction starts
figs = {}
for c, s in excess.items():
    al = za.ph.align(ph_use, s, window_s=PH_WINDOW_S)
    if len(al) < 3:
        print(lab[c], ": fewer than 3 pH samples overlap with recordings"); continue
    fig = zp.plot_ph_scatter(al, label=f"{lab[c]} excess")
    fig.axes[0].set_title(lab[c], loc="left")
    zp.save(fig, cfg, f"level_vs_pH_ch{c}")
    rho = al["pH"].corr(al["acoustic_dB"], method="spearman")
    print(f"{lab[c]}: n={len(al)}, Spearman rho={rho:.3f}")

### Lag between pH and sound

If the pH probe sits away from the sample, or responds slowly, pH lags the acoustics. The lag with the
largest |r| is a *hint*, not a measurement: with smooth, autocorrelated series many lags correlate well.
Only lags covered by recordings count (see `n`).

In [ ]:
c = next(iter(excess))
corr = za.ph.lagged_correlation(dsp.db(excess[c]).rename("ac"), ph_use["pH"], dt_s=DT_S, max_lag_s=300)
fig = zp.plot_lag(corr, title=f"{lab[c]} excess level vs pH")
corr.loc[corr.r.abs().idxmax()] if corr.r.notna().any() else corr

## Export the aligned table (for your own plots, Excel, etc.)

In [ ]:
out = pd.DataFrame({"pH": ph["pH"]})
for c, s in excess.items():
    out[f"ch{c}_{BAND}_excess_dB"] = za.ph.acoustic_at(ph.index, s, PH_WINDOW_S).to_numpy()
out.to_csv(cfg.figures_path / "ph_acoustic_aligned.csv")
out.dropna().head()